# Solutions — Wanderbricks Advanced BI (Hard 08)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

bookings    = spark.table("samples.wanderbricks.bookings")
users       = spark.table("samples.wanderbricks.users")
properties  = spark.table("samples.wanderbricks.properties")
reviews     = spark.table("samples.wanderbricks.reviews")
payments    = spark.table("samples.wanderbricks.payments")
hosts       = spark.table("samples.wanderbricks.hosts")
clickstream = spark.table("samples.wanderbricks.clickstream")


## Solution 1 — Dynamic Pricing Analysis

In [ ]:
result_1 = (
    bookings
    .filter(F.col("status") == "confirmed")
    .withColumn("nights", F.datediff("check_out", "check_in"))
    .filter(F.col("nights") > 0)
    .withColumn("charged_per_night", F.col("total_amount") / F.col("nights"))
    .groupBy("property_id")
    .agg(
        F.round(F.avg("charged_per_night"), 2).alias("avg_charged_per_night"),
        F.count("*").alias("booking_count"),
    )
    .join(properties.select("property_id", "title", "property_type", "base_price"), on="property_id")
    .withColumn("price_premium_pct",
        F.round((F.col("avg_charged_per_night") - F.col("base_price")) / F.col("base_price") * 100, 2)
    )
    .select("property_id", "title", "property_type", "base_price",
            "avg_charged_per_night", "price_premium_pct", "booking_count")
    .orderBy(F.col("price_premium_pct").desc())
)


## Solution 2 — Review Score Trajectory

In [ ]:
w_early  = Window.partitionBy("property_id").orderBy("created_at")
w_recent = Window.partitionBy("property_id").orderBy(F.col("created_at").desc())

with_rank = (
    reviews
    .filter(F.col("is_deleted") == False)
    .withColumn("early_rank",  F.row_number().over(w_early))
    .withColumn("recent_rank", F.row_number().over(w_recent))
)

early_avg = (
    with_rank.filter(F.col("early_rank")  <= 5)
    .groupBy("property_id")
    .agg(F.round(F.avg("rating"), 2).alias("early_avg_rating"))
)

recent_avg = (
    with_rank.filter(F.col("recent_rank") <= 5)
    .groupBy("property_id")
    .agg(F.round(F.avg("rating"), 2).alias("recent_avg_rating"))
)

total_cnt = (
    reviews.filter(F.col("is_deleted") == False)
    .groupBy("property_id")
    .agg(F.count("*").alias("total_reviews"))
)

result_2 = (
    early_avg
    .join(recent_avg, on="property_id")
    .join(total_cnt,  on="property_id")
    .withColumn("rating_trend",
        F.when(F.col("recent_avg_rating") - F.col("early_avg_rating") >  0.2, "improving")
         .when(F.col("early_avg_rating")  - F.col("recent_avg_rating") >  0.2, "declining")
         .otherwise("stable")
    )
    .select("property_id", "early_avg_rating", "recent_avg_rating", "rating_trend", "total_reviews")
    .orderBy("property_id")
)


## Solution 3 — Superhost Identification

In [ ]:
avg_rating = (
    reviews.filter(F.col("is_deleted") == False)
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(F.round(F.avg("rating"), 2).alias("avg_rating"))
)

booking_stats = (
    bookings
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(
        F.sum(F.when(F.col("status") == "confirmed",  1).otherwise(0)).alias("confirmed_bookings"),
        F.sum(F.when(F.col("status") == "cancelled",  1).otherwise(0)).alias("cancelled_count"),
        F.count("*").alias("total_bookings"),
    )
    .withColumn("cancellation_rate_pct",
        F.round(F.col("cancelled_count") / F.col("total_bookings") * 100, 2)
    )
)

result_3 = (
    hosts
    .join(avg_rating,    on="host_id", how="left")
    .join(booking_stats, on="host_id", how="left")
    .withColumn("is_superhost",
        (F.col("is_verified") == True) &
        (F.col("avg_rating") >= 4.5) &
        (F.col("confirmed_bookings") >= 10) &
        (F.col("cancellation_rate_pct") < 5)
    )
    .select("host_id", "name", "avg_rating", "confirmed_bookings",
            "cancellation_rate_pct", "is_superhost")
)


## Solution 4 — Cross-Property User Behaviour

In [ ]:
result_4 = (
    bookings
    .join(properties.select("property_id", "property_type"), on="property_id")
    .groupBy("user_id")
    .agg(
        F.collect_set("property_type").alias("property_types_booked"),
        F.countDistinct("property_type").alias("type_count"),
        F.round(F.sum("total_amount"), 2).alias("total_spend"),
    )
    .filter(F.col("type_count") > 1)
    .join(users.select("user_id", "name"), on="user_id")
    .select("user_id", "name", "property_types_booked", "type_count", "total_spend")
    .orderBy(F.col("total_spend").desc())
)


## Solution 5 — Seasonal Demand Analysis

In [ ]:
w = Window.partitionBy("property_type").orderBy(F.col("booking_count").desc())

result_5 = (
    bookings
    .join(properties.select("property_id", "property_type"), on="property_id")
    .withColumn("month", F.month("check_in"))
    .groupBy("property_type", "month")
    .agg(
        F.count("*").alias("booking_count"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    )
    .withColumn("demand_rank", F.rank().over(w))
    .orderBy("property_type", "demand_rank")
)


## Solution 6 — Payment vs Booking Amount Reconciliation

In [ ]:
total_paid = (
    payments
    .filter(F.col("status") == "completed")
    .groupBy("booking_id")
    .agg(F.round(F.sum("amount"), 2).alias("total_paid"))
)

result_6 = (
    bookings
    .join(total_paid, on="booking_id", how="left")
    .withColumn("total_paid",    F.coalesce("total_paid", F.lit(0.0)))
    .withColumn("booking_amount", F.col("total_amount"))
    .withColumn("variance",       F.round(F.col("total_paid") - F.col("booking_amount"), 2))
    .withColumn("variance_pct",   F.round(F.abs(F.col("variance")) / F.col("booking_amount") * 100, 2))
    .withColumn("is_reconciled",  F.col("variance_pct") <= 1.0)
    .select("booking_id", "booking_amount", "total_paid", "variance", "variance_pct", "is_reconciled")
)


## Solution 7 — Property Recommendation Features

In [ ]:
prop_reviews = (
    reviews.filter(F.col("is_deleted") == False)
    .groupBy("property_id")
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("*").alias("review_count"),
    )
)

prop_bookings = (
    bookings.filter(F.col("status") == "confirmed")
    .withColumn("stay_nights", F.datediff("check_out", "check_in"))
    .groupBy("property_id")
    .agg(
        F.count("*").alias("booking_count"),
        F.round(F.sum("stay_nights") / 365 * 100, 2).alias("occupancy_rate"),
        F.round(F.avg("total_amount"), 2).alias("avg_booking_value"),
        F.round(F.avg("stay_nights"), 2).alias("avg_stay_nights"),
    )
)

result_7 = (
    properties
    .join(prop_reviews,  on="property_id", how="left")
    .join(prop_bookings, on="property_id", how="left")
    .select("property_id", "title", "property_type",
            "avg_rating", "review_count", "booking_count",
            "occupancy_rate", "avg_booking_value", "avg_stay_nights")
)


## Solution 8 — Host Revenue Concentration (Gini-like)

In [ ]:
host_rev = (
    bookings.filter(F.col("status") == "confirmed")
    .join(properties.select("property_id", "host_id"), on="property_id")
    .groupBy("host_id")
    .agg(F.round(F.sum("total_amount"), 2).alias("total_revenue"))
)

total_rev   = host_rev.agg(F.sum("total_revenue")).collect()[0][0]
total_hosts = host_rev.count()

w_rank = Window.orderBy(F.col("total_revenue").desc())
w_cum  = Window.orderBy(F.col("total_revenue").desc()).rowsBetween(Window.unboundedPreceding, 0)

result_8 = (
    host_rev
    .withColumn("revenue_rank",          F.rank().over(w_rank))
    .withColumn("host_percentile",        F.round(F.col("revenue_rank") / total_hosts * 100, 2))
    .withColumn("cumulative_revenue_pct", F.round(F.sum("total_revenue").over(w_cum) / total_rev * 100, 2))
    .select("host_id", "total_revenue", "revenue_rank", "cumulative_revenue_pct", "host_percentile")
    .orderBy("revenue_rank")
)
